In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
import pickle

In [ ]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.dropna(inplace=True)
df.drop("customerID", axis=1, inplace=True)

In [22]:
y = df["Churn"].map({"Yes": 1, "No": 0})
X = df.drop("Churn", axis=1)

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [24]:
def engineer(df):
    df = df.copy()
    bins = [0,12,24,36,48,60,72]
    labels = ["0-1yr","1-2yr","2-3yr","3-4yr","4-5yr","5+yr"]
    df["tenure_group"] = pd.cut(df["tenure"], bins=bins, labels=labels)
    df["HighRisk_Contract"] = (df["Contract"] == "Month-to-month").astype(int)
    df["NoSupport"] = (
        (df["TechSupport"] == "No") &
        (df["OnlineSecurity"] == "No")
    ).astype(int)
    df["FiberUser"] = (df["InternetService"] == "Fiber optic").astype(int)
    return df
X_train = engineer(X_train)
X_test = engineer(X_test)

In [25]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

In [26]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_split=5,
        class_weight="balanced",
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="logloss",
        use_label_encoder=False
    )
}

In [35]:
results = []
for name, model in models.items():
    print('\n')
    print(f"Training: {name}")
    print ('\n')
    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:,1]
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    cm = confusion_matrix(y_test, y_pred)
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")
    print("Confusion Matrix:")
    print(cm)
    results.append({
        "Model": name,
        "ROC-AUC": auc,
        "Pipeline": pipeline
    })



Training: Logistic Regression


Accuracy : 0.7264
Precision: 0.4909
Recall   : 0.7941
F1 Score : 0.6067
ROC-AUC  : 0.8335
Confusion Matrix:
[[725 308]
 [ 77 297]]


Training: Random Forest


Accuracy : 0.7569
Precision: 0.5314
Recall   : 0.7246
F1 Score : 0.6131
ROC-AUC  : 0.8309
Confusion Matrix:
[[794 239]
 [103 271]]


Training: XGBoost


Accuracy : 0.7370
Precision: 0.5035
Recall   : 0.7674
F1 Score : 0.6081
ROC-AUC  : 0.8305
Confusion Matrix:
[[750 283]
 [ 87 287]]


In [32]:
best_model = sorted(results, key=lambda x: x["ROC-AUC"], reverse=True)[0]
print("BEST MODEL:", best_model["Model"])

BEST MODEL: Logistic Regression


In [36]:
pickle.dump(best_model["Pipeline"], open("model.sav", "wb"))
print("\nModel saved successfully!")


Model saved successfully!
